# CET6 听力音频批量转录 (Faster-Whisper)

将无PDF原文的听力音频通过Whisper转为文本。

In [1]:
# Cell 1: 安装依赖
!pip install -q faster-whisper
!apt-get -qq -y install ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.2 MB/s eta 0:00:00


In [2]:
# Cell 2: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, re, shutil

DRIVE_BASE = '/content/drive/MyDrive/CET6-Resources'

Mounted at /content/drive


In [3]:
# Cell 3: 定义待处理音频列表
# 这些音频的解析PDF中不含听力原文，需要Whisper转录

audio_list = [
    ('CET6_2016.12', 'CET6_2016.12_set3_listening.mp3', 'CET6_2016.12_set3_transcript.md'),
    ('CET6_2018.06', 'CET6_2018.06_set1_listening.mp3', 'CET6_2018.06_set1_transcript.md'),
    ('CET6_2018.06', 'CET6_2018.06_set2_listening.mp3', 'CET6_2018.06_set2_transcript.md'),
    ('CET6_2018.12', 'CET6_2018.12_set1_listening.mp3', 'CET6_2018.12_set1_transcript.md'),
    ('CET6_2018.12', 'CET6_2018.12_set2_listening.mp3', 'CET6_2018.12_set2_transcript.md'),
    ('CET6_2019.12', 'CET6_2019.12_set3_listening.mp3', 'CET6_2019.12_set3_transcript.md'),
    ('CET6_2020.07', 'CET6_2020.07_listening.m4a', 'CET6_2020.07_transcript.md'),
    ('CET6_2020.09', 'CET6_2020.09_listening.m4a', 'CET6_2020.09_transcript.md'),
    ('CET6_2021.06', 'CET6_2021.06_set1_listening.mp3', 'CET6_2021.06_set1_transcript.md'),
    ('CET6_2021.06', 'CET6_2021.06_set2_listening.mp3', 'CET6_2021.06_set2_transcript.md'),
    ('CET6_2021.12', 'CET6_2021.12_set1_listening.mp3', 'CET6_2021.12_set1_transcript.md'),
    ('CET6_2021.12', 'CET6_2021.12_set2_listening.mp3', 'CET6_2021.12_set2_transcript.md'),
]

# 验证文件存在
found, missing = [], []
for dir_name, audio_name, _ in audio_list:
    path = os.path.join(DRIVE_BASE, dir_name, audio_name)
    if os.path.exists(path):
        found.append(audio_name)
    else:
        missing.append(f'{dir_name}/{audio_name}')

print(f'找到 {len(found)}/{len(audio_list)} 个音频')
if missing:
    print('未找到:')
    for m in missing:
        print(f'  ✗ {m}')

找到 12/12 个音频


In [4]:
# Cell 4: 加载模型
from faster_whisper import WhisperModel

model = WhisperModel(
    "large-v2",
    device="cuda",
    compute_type="float16"
)
print('模型加载完成')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


模型加载完成


In [5]:
# Cell 5: 批量转录

def transcribe_audio(audio_path):
    """转录单个音频文件，返回带时间戳的文本段落"""
    segments, info = model.transcribe(
        audio_path,
        language="en",
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=600),
    )

    results = []
    for seg in segments:
        results.append({
            'start': seg.start,
            'end': seg.end,
            'text': seg.text.strip()
        })
    return results


def format_transcript(segments):
    """将whisper段落合并为连续文本，按停顿分段"""
    if not segments:
        return ''

    paragraphs = []
    current_para = []
    prev_end = 0

    for seg in segments:
        # 超过3秒停顿视为新段落
        if current_para and (seg['start'] - prev_end) > 3.0:
            paragraphs.append(' '.join(current_para))
            current_para = []
        current_para.append(seg['text'])
        prev_end = seg['end']

    if current_para:
        paragraphs.append(' '.join(current_para))

    return '\n\n'.join(paragraphs)


# 批量处理
all_results = {}

for i, (dir_name, audio_name, target_name) in enumerate(audio_list):
    audio_path = os.path.join(DRIVE_BASE, dir_name, audio_name)
    if not os.path.exists(audio_path):
        print(f'[{i+1}/{len(audio_list)}] 跳过（不存在）: {audio_name}')
        continue

    print(f'[{i+1}/{len(audio_list)}] 转录中: {audio_name} ...', end=' ')

    try:
        segments = transcribe_audio(audio_path)
        transcript = format_transcript(segments)
        all_results[target_name] = {
            'dir': dir_name,
            'text': transcript,
            'segments': len(segments)
        }
        print(f'完成 ({len(segments)} segments)')
    except Exception as e:
        print(f'失败: {e}')

print(f'\n转录完成: {len(all_results)}/{len(audio_list)}')

[1/12] 转录中: CET6_2016.12_set3_listening.mp3 ... 完成 (242 segments)
[2/12] 转录中: CET6_2018.06_set1_listening.mp3 ... 完成 (286 segments)
[3/12] 转录中: CET6_2018.06_set2_listening.mp3 ... 完成 (289 segments)
[4/12] 转录中: CET6_2018.12_set1_listening.mp3 ... 完成 (282 segments)
[5/12] 转录中: CET6_2018.12_set2_listening.mp3 ... 完成 (301 segments)
[6/12] 转录中: CET6_2019.12_set3_listening.mp3 ... 完成 (252 segments)
[7/12] 转录中: CET6_2020.07_listening.m4a ... 完成 (302 segments)
[8/12] 转录中: CET6_2020.09_listening.m4a ... 完成 (284 segments)
[9/12] 转录中: CET6_2021.06_set1_listening.mp3 ... 完成 (434 segments)
[10/12] 转录中: CET6_2021.06_set2_listening.mp3 ... 完成 (214 segments)
[11/12] 转录中: CET6_2021.12_set1_listening.mp3 ... 完成 (295 segments)
[12/12] 转录中: CET6_2021.12_set2_listening.mp3 ... 完成 (396 segments)

转录完成: 12/12


In [6]:
# Cell 6: 保存结果到 Google Drive

for target_name, data in all_results.items():
    dst_dir = os.path.join(DRIVE_BASE, data['dir'])
    dst_path = os.path.join(dst_dir, target_name)

    with open(dst_path, 'w', encoding='utf-8') as f:
        f.write(data['text'])

    print(f'  ✓ {data["dir"]}/{target_name} ({data["segments"]} segments)')

print(f'\n共保存 {len(all_results)} 个文件到 Drive')

  ✓ CET6_2016.12/CET6_2016.12_set3_transcript.md (242 segments)
  ✓ CET6_2018.06/CET6_2018.06_set1_transcript.md (286 segments)
  ✓ CET6_2018.06/CET6_2018.06_set2_transcript.md (289 segments)
  ✓ CET6_2018.12/CET6_2018.12_set1_transcript.md (282 segments)
  ✓ CET6_2018.12/CET6_2018.12_set2_transcript.md (301 segments)
  ✓ CET6_2019.12/CET6_2019.12_set3_transcript.md (252 segments)
  ✓ CET6_2020.07/CET6_2020.07_transcript.md (302 segments)
  ✓ CET6_2020.09/CET6_2020.09_transcript.md (284 segments)
  ✓ CET6_2021.06/CET6_2021.06_set1_transcript.md (434 segments)
  ✓ CET6_2021.06/CET6_2021.06_set2_transcript.md (214 segments)
  ✓ CET6_2021.12/CET6_2021.12_set1_transcript.md (295 segments)
  ✓ CET6_2021.12/CET6_2021.12_set2_transcript.md (396 segments)

共保存 12 个文件到 Drive


In [7]:
# Cell 7: 预览某个转录结果
# 修改 idx 查看不同文件
idx = 0
target_name = list(all_results.keys())[idx]
print(f'=== {target_name} ===\n')
print(all_results[target_name]['text'][:3000])

=== CET6_2016.12_set3_transcript.md ===

College English Test, Band 4, Part 2, Listening Comprehension, Section A, Directions. In this section, you will hear three news reports. At the end of each news report, you will hear two or three questions. Both the news report and the questions will be spoken only once. After you hear a question, you must choose the best answer from the four choices marked A, B, C, and D. Then mark the corresponding letter on Answer Sheet 1 with a single line through the center. Questions 1 and 2 will be based on the following news item. Fifteen passengers were burned alive when a passenger bus was set on fire by an unknown min in Pakistan's southwestern Balochistan province, an official said. Nasir Ahmed Nasir, deputy commissioner in the city of Sibi, said four men on motorcycles initially opened fire on a parked passenger bus at a hotel. The armed men then came close to the bus and splashed petrol, setting it on fire, he said. He said most of the dead were ei